In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Marketplace Trust & Safety - Seller Risk Demo\n",
    "This notebook demonstrates how a trust-and-safety analyst\n",
    "would explore the curated seller risk datasets."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from pyspark.sql import SparkSession\n",
    "\n",
    "spark = SparkSession.builder \\\n",
    "    .appName('SellerRiskDemo') \\\n",
    "    .enableHiveSupport() \\\n",
    "    .getOrCreate()\n",
    "\n",
    "print('Spark session started')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load the Curated Seller Risk Table"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "seller_risk = spark.read.parquet(\n",
    "    'hdfs://namenode:9000/data/curated/seller_risk/'\n",
    ")\n",
    "seller_risk.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Show High Risk Sellers"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from pyspark.sql.functions import col\n",
    "\n",
    "high_risk = seller_risk.filter(col('risk_label') == 'HIGH') \\\n",
    "    .orderBy(col('risk_score').desc())\n",
    "\n",
    "high_risk.select(\n",
    "    'seller_id', 'seller_name', 'region',\n",
    "    'risk_score', 'risk_label',\n",
    "    'return_rate', 'high_severity_complaints'\n",
    ").show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Show Fraud Patterns"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fraud = spark.read.parquet(\n",
    "    'hdfs://namenode:9000/data/curated/fraud_patterns/'\n",
    ")\n",
    "fraud.show(truncate=False)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Risk Score Distribution"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "df = seller_risk.select(\n",
    "    'seller_name', 'risk_score', 'risk_label'\n",
    ").toPandas()\n",
    "\n",
    "colors = df['risk_label'].map({\n",
    "    'HIGH': 'red',\n",
    "    'MEDIUM': 'orange',\n",
    "    'LOW': 'green'\n",
    "})\n",
    "\n",
    "plt.figure(figsize=(10, 5))\n",
    "plt.bar(df['seller_name'], df['risk_score'], color=colors)\n",
    "plt.title('Seller Risk Scores')\n",
    "plt.xlabel('Seller')\n",
    "plt.ylabel('Risk Score')\n",
    "plt.xticks(rotation=45)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Return Rate vs Complaints (Scatter Plot)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df2 = seller_risk.select(\n",
    "    'seller_name', 'return_rate',\n",
    "    'total_complaints', 'risk_label'\n",
    ").toPandas()\n",
    "\n",
    "colors2 = df2['risk_label'].map({\n",
    "    'HIGH': 'red',\n",
    "    'MEDIUM': 'orange',\n",
    "    'LOW': 'green'\n",
    "})\n",
    "\n",
    "plt.figure(figsize=(8, 5))\n",
    "plt.scatter(\n",
    "    df2['return_rate'],\n",
    "    df2['total_complaints'],\n",
    "    c=colors2, s=100\n",
    ")\n",
    "\n",
    "for i, row in df2.iterrows():\n",
    "    plt.annotate(\n",
    "        row['seller_name'],\n",
    "        (row['return_rate'], row['total_complaints']),\n",
    "        textcoords='offset points',\n",
    "        xytext=(5, 5)\n",
    "    )\n",
    "\n",
    "plt.title('Return Rate vs Total Complaints')\n",
    "plt.xlabel('Return Rate')\n",
    "plt.ylabel('Total Complaints')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Investigate a Specific Seller"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "seller_id = 'S55'\n",
    "\n",
    "print(f'=== Risk Profile for Seller {seller_id} ===')\n",
    "seller_risk.filter(\n",
    "    col('seller_id') == seller_id\n",
    ").show(truncate=False)\n",
    "\n",
    "print(f'=== Fraud Patterns for Seller {seller_id} ===')\n",
    "fraud.filter(\n",
    "    col('seller_id') == seller_id\n",
    ").show(truncate=False)"
   ]
  }
 ]
}